# Install dependencies

In [1]:
import sys
import subprocess
import importlib.util

packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "PIL": "pillow",
    "tqdm": "tqdm",
    "imgutils": "dghs-imgutils",
}

missing = [pip_name for module_name, pip_name in packages.items() if importlib.util.find_spec(module_name) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Dependency check completed.")

Dependency check completed.


# Imports

In [2]:
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from datetime import datetime
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import json
import os
import sys
import shutil
import platform
import hashlib
import math
import warnings
import traceback
import gc

Image.MAX_IMAGE_PIXELS = 300_000_000
warnings.simplefilter("error", Image.DecompressionBombWarning)

try:
    from imgutils.tagging import get_pixai_tags
except Exception as exc:
    raise RuntimeError(
        "Could not import get_pixai_tags from dghs-imgutils. "
        "Restart the kernel after installation and run again."
    ) from exc

try:
    from imgutils.tagging import get_wd14_tags
    WD14_AVAILABLE = True
except Exception:
    get_wd14_tags = None
    WD14_AVAILABLE = False

print("Imports completed.")
print(f"WD14 fallback available: {WD14_AVAILABLE}")

Imports completed.
WD14 fallback available: True


# Configuration

In [3]:
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()

PREVIOUS_CLUSTERING_RUN_DIR = None
PREVIOUS_MATERIALIZATION_RUN_DIR = None

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = PROJECT_DIR / "runs" / f"06_cluster_naming_{RUN_ID}"

REPORTS_DIR = RUN_DIR / "reports"
TABLES_DIR = RUN_DIR / "tables"
LOGS_DIR = RUN_DIR / "logs"
CONTACT_SHEETS_DIR = RUN_DIR / "contact_sheets"

for directory in [RUN_DIR, REPORTS_DIR, TABLES_DIR, LOGS_DIR, CONTACT_SHEETS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PRIMARY_TAGGER = "pixai"
PIXAI_MODEL_NAME = "v0.9"
WD14_MODEL_NAME = "SwinV2_v3"

USE_WD14_FALLBACK = True

MIN_CLUSTER_SIZE_TO_NAME = 2
MAX_IMAGES_PER_CLUSTER_TO_TAG = 12

MIN_CHARACTER_SCORE = 0.70
MIN_NAME_SHARE = 0.35
MIN_WEIGHTED_SCORE = 0.55
MIN_TOP_MARGIN = 0.08

IGNORE_REVIEW_FOLDERS_FOR_NAMING = False
TAG_ONLY_NON_NOISE_CLUSTERS = True

CREATE_RENAME_PLAN = True
CREATE_NAMED_OUTPUT = True

NAMED_OUTPUT_MODE = "hardlink"
ALLOW_HARDLINK_FALLBACK_TO_COPY = True
ON_EXISTING = "error"

NAMED_OUTPUT_DIR = PROJECT_DIR / "organized_output" / f"anime_named_{RUN_ID}"

CONTACT_SHEET_THUMB_SIZE = 170
CONTACT_SHEET_COLUMNS = 6
CONTACT_SHEET_MAX_IMAGES_PER_CLUSTER = 24
MAX_CONTACT_SHEETS = 250

VALID_NAMED_OUTPUT_MODES = {"hardlink", "copy", "symlink"}
VALID_ON_EXISTING = {"error", "skip", "overwrite"}

if NAMED_OUTPUT_MODE not in VALID_NAMED_OUTPUT_MODES:
    raise ValueError(f"Invalid NAMED_OUTPUT_MODE: {NAMED_OUTPUT_MODE}")

if ON_EXISTING not in VALID_ON_EXISTING:
    raise ValueError(f"Invalid ON_EXISTING: {ON_EXISTING}")

if CREATE_NAMED_OUTPUT:
    NAMED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Run directory: {RUN_DIR}")
print(f"Primary tagger: {PRIMARY_TAGGER}")
print(f"PixAI model: {PIXAI_MODEL_NAME}")
print(f"Create named output: {CREATE_NAMED_OUTPUT}")

Project directory: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline
Run directory: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/06_cluster_naming_20260523_221209
Primary tagger: pixai
PixAI model: v0.9
Create named output: False


# Locate previous clustering and materialization runs


In [4]:
def find_latest_valid_run(project_dir, pattern, required_relative_paths):
    runs_dir = Path(project_dir) / "runs"
    candidates = sorted(
        [p for p in runs_dir.glob(pattern) if p.is_dir()],
        key=lambda p: p.name,
    )

    valid_candidates = []

    for candidate in candidates:
        if all((candidate / rel_path).exists() for rel_path in required_relative_paths):
            valid_candidates.append(candidate)

    if not valid_candidates:
        required_text = ", ".join(str(p) for p in required_relative_paths)
        raise FileNotFoundError(
            f"No valid run matching {pattern} was found in {runs_dir}. "
            f"Required files: {required_text}"
        )

    return valid_candidates[-1]


PREVIOUS_CLUSTERING_RUN_DIR = globals().get("PREVIOUS_CLUSTERING_RUN_DIR", None)

if PREVIOUS_CLUSTERING_RUN_DIR is None:
    PREVIOUS_CLUSTERING_RUN_DIR = find_latest_valid_run(
        PROJECT_DIR,
        "04_hdbscan_clustering_*",
        [
            Path("tables") / "cluster_manifest.csv",
            Path("tables") / "cluster_summary.csv",
            Path("tables") / "folder_assignment_manifest.csv",
        ],
    )
else:
    PREVIOUS_CLUSTERING_RUN_DIR = Path(PREVIOUS_CLUSTERING_RUN_DIR).expanduser().resolve()

PREVIOUS_MATERIALIZATION_RUN_DIR = globals().get("PREVIOUS_MATERIALIZATION_RUN_DIR", None)

if PREVIOUS_MATERIALIZATION_RUN_DIR is None:
    try:
        PREVIOUS_MATERIALIZATION_RUN_DIR = find_latest_valid_run(
            PROJECT_DIR,
            "05_folder_materialization_*",
            [
                Path("tables") / "materialization_plan.csv",
                Path("tables") / "materialization_result.csv",
            ],
        )
    except FileNotFoundError:
        PREVIOUS_MATERIALIZATION_RUN_DIR = None
else:
    PREVIOUS_MATERIALIZATION_RUN_DIR = Path(PREVIOUS_MATERIALIZATION_RUN_DIR).expanduser().resolve()

cluster_manifest_path = PREVIOUS_CLUSTERING_RUN_DIR / "tables" / "cluster_manifest.csv"
cluster_summary_path = PREVIOUS_CLUSTERING_RUN_DIR / "tables" / "cluster_summary.csv"
folder_assignment_path = PREVIOUS_CLUSTERING_RUN_DIR / "tables" / "folder_assignment_manifest.csv"

cluster_manifest_df = pd.read_csv(cluster_manifest_path)
cluster_summary_df = pd.read_csv(cluster_summary_path)
folder_assignment_df = pd.read_csv(folder_assignment_path)

if PREVIOUS_MATERIALIZATION_RUN_DIR is not None:
    materialization_plan_path = PREVIOUS_MATERIALIZATION_RUN_DIR / "tables" / "materialization_plan.csv"
    materialization_result_path = PREVIOUS_MATERIALIZATION_RUN_DIR / "tables" / "materialization_result.csv"
    materialization_plan_df = pd.read_csv(materialization_plan_path)
    materialization_result_df = pd.read_csv(materialization_result_path)
else:
    materialization_plan_path = None
    materialization_result_path = None
    materialization_plan_df = pd.DataFrame()
    materialization_result_df = pd.DataFrame()

print(f"Previous clustering run: {PREVIOUS_CLUSTERING_RUN_DIR}")
print(f"Previous materialization run: {PREVIOUS_MATERIALIZATION_RUN_DIR}")
print(f"Cluster manifest rows: {len(cluster_manifest_df)}")
print(f"Cluster summary rows: {len(cluster_summary_df)}")
print(f"Folder assignment rows: {len(folder_assignment_df)}")
print(f"Materialization plan rows: {len(materialization_plan_df)}")


Previous clustering run: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/04_hdbscan_clustering_20260523_214927
Previous materialization run: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/05_folder_materialization_20260523_220036
Cluster manifest rows: 15705
Cluster summary rows: 189
Folder assignment rows: 15705
Materialization plan rows: 15705


# Helper functions

In [5]:
def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")


def normalize_path_string(value):
    return Path(str(value)).expanduser().resolve().as_posix()


def stable_short_hash(value, length=12):
    return hashlib.sha256(str(value).encode("utf-8", errors="replace")).hexdigest()[:length]


def safe_bool(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def safe_float(value, default=np.nan):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def sanitize_filename_component(value, fallback="item", max_length=120):
    value = str(value).strip()

    if not value:
        value = fallback

    forbidden = '<>:"/\\|?*'
    cleaned = "".join("_" if ch in forbidden else ch for ch in value)
    cleaned = "".join(ch if ch.isprintable() else "_" for ch in cleaned)
    cleaned = cleaned.strip().strip(".")
    cleaned = "_".join(cleaned.split())

    if not cleaned:
        cleaned = fallback

    return cleaned[:max_length]


def normalize_tag_name(tag):
    tag = str(tag).strip()
    tag = tag.replace("\\", "")
    tag = tag.replace("/", "_")
    tag = tag.replace(" ", "_")
    tag = tag.strip("_")
    return tag


def display_tag_name(tag):
    return normalize_tag_name(tag)


def safe_json_dumps(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def parse_dict_like(value):
    if isinstance(value, dict):
        return value

    if pd.isna(value):
        return {}

    if isinstance(value, str):
        value = value.strip()
        if not value:
            return {}
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            return {}

    return {}


def validate_image_file(path):
    path = Path(path)

    if not path.exists():
        return False, "FileNotFoundError", "Image file does not exist."

    if not path.is_file():
        return False, "NotAFileError", "Path exists but is not a regular file."

    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img.convert("RGB"))
            width, height = img.size

        if width <= 0 or height <= 0:
            return False, "InvalidImageSize", "Image has non-positive dimensions."

        return True, None, None

    except Exception as exc:
        return False, type(exc).__name__, str(exc)


def select_cluster_samples(cluster_df, max_images):
    cluster_df = cluster_df.copy()

    if "distance_to_centroid" not in cluster_df.columns:
        cluster_df["distance_to_centroid"] = np.nan

    if "cluster_probability" not in cluster_df.columns:
        cluster_df["cluster_probability"] = 1.0

    if "outlier_score" not in cluster_df.columns:
        cluster_df["outlier_score"] = 0.0

    cluster_df["distance_to_centroid_sort"] = cluster_df["distance_to_centroid"].fillna(cluster_df["distance_to_centroid"].median())
    cluster_df["cluster_probability_sort"] = cluster_df["cluster_probability"].fillna(0.0)
    cluster_df["outlier_score_sort"] = cluster_df["outlier_score"].fillna(1.0)

    representative_df = cluster_df.sort_values(
        ["cluster_probability_sort", "outlier_score_sort", "distance_to_centroid_sort"],
        ascending=[False, True, True],
    ).head(max_images)

    if len(representative_df) < max_images:
        remaining_df = cluster_df[~cluster_df["embedding_row"].isin(representative_df["embedding_row"])].copy()
        extra_df = remaining_df.sort_values(
            ["distance_to_centroid_sort", "cluster_probability_sort"],
            ascending=[True, False],
        ).head(max_images - len(representative_df))
        representative_df = pd.concat([representative_df, extra_df], axis=0)

    representative_df = representative_df.drop_duplicates(subset=["embedding_row"], keep="first")
    representative_df = representative_df.head(max_images).copy()

    return representative_df.drop(columns=[
        col for col in ["distance_to_centroid_sort", "cluster_probability_sort", "outlier_score_sort"]
        if col in representative_df.columns
    ])


def extract_pixai_tags(image_path):
    general, character, ips, ips_mapping = get_pixai_tags(
        image_path,
        model_name=PIXAI_MODEL_NAME,
        fmt=("general", "character", "ips", "ips_mapping"),
    )

    if general is None:
        general = {}

    if character is None:
        character = {}

    if ips is None:
        ips = []

    if ips_mapping is None:
        ips_mapping = {}

    general = {normalize_tag_name(k): float(v) for k, v in dict(general).items()}
    character = {normalize_tag_name(k): float(v) for k, v in dict(character).items()}

    return {
        "tagger": "pixai",
        "general_tags": general,
        "character_tags": character,
        "ips": [normalize_tag_name(x) for x in list(ips)],
        "ips_mapping": {
            normalize_tag_name(k): [normalize_tag_name(x) for x in v]
            for k, v in dict(ips_mapping).items()
        },
    }


def extract_wd14_tags(image_path):
    if not WD14_AVAILABLE:
        raise RuntimeError("WD14 fallback is not available in this environment.")

    result = get_wd14_tags(
        image_path,
        model_name=WD14_MODEL_NAME,
        fmt=("rating", "general", "character"),
    )

    if not isinstance(result, tuple) or len(result) != 3:
        raise ValueError(f"Unexpected WD14 output format: {type(result)}")

    rating, general, character = result

    if general is None:
        general = {}

    if character is None:
        character = {}

    general = {normalize_tag_name(k): float(v) for k, v in dict(general).items()}
    character = {normalize_tag_name(k): float(v) for k, v in dict(character).items()}

    return {
        "tagger": "wd14",
        "rating_tags": {normalize_tag_name(k): float(v) for k, v in dict(rating or {}).items()},
        "general_tags": general,
        "character_tags": character,
        "ips": [],
        "ips_mapping": {},
    }


def extract_tags_with_fallback(image_path):
    try:
        return extract_pixai_tags(image_path), None, None
    except Exception as pixai_exc:
        if USE_WD14_FALLBACK and WD14_AVAILABLE:
            try:
                return extract_wd14_tags(image_path), type(pixai_exc).__name__, str(pixai_exc)
            except Exception as wd14_exc:
                return None, type(wd14_exc).__name__, str(wd14_exc)

        return None, type(pixai_exc).__name__, str(pixai_exc)


def create_contact_sheet(image_paths, output_path, thumb_size=170, columns=6, title=None):
    valid_paths = [Path(p) for p in image_paths if pd.notna(p) and Path(p).exists()]

    if not valid_paths:
        return False

    rows = int(math.ceil(len(valid_paths) / columns))
    title_height = 34 if title else 0

    sheet_width = columns * thumb_size
    sheet_height = rows * thumb_size + title_height

    sheet = Image.new("RGB", (sheet_width, sheet_height), "white")
    draw = ImageDraw.Draw(sheet)

    if title:
        draw.rectangle((0, 0, sheet_width, title_height), fill=(245, 245, 245))
        draw.text((10, 9), title[:180], fill=(20, 20, 20))

    for idx, path in enumerate(valid_paths):
        try:
            with Image.open(path) as img:
                img = ImageOps.exif_transpose(img.convert("RGB"))
                img.thumbnail((thumb_size, thumb_size), Image.Resampling.LANCZOS)

                x = (idx % columns) * thumb_size + (thumb_size - img.width) // 2
                y = title_height + (idx // columns) * thumb_size + (thumb_size - img.height) // 2

                sheet.paste(img, (x, y))
        except Exception:
            continue

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(output_path, format="JPEG", quality=92, optimize=True)

    return True


def remove_existing_file(path):
    path = Path(path)

    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        raise IsADirectoryError(f"Destination exists and is not a regular file or symlink: {path}")


def materialize_one_file(source_path, destination_path, mode, on_existing, allow_hardlink_fallback_to_copy):
    source_path = Path(source_path)
    destination_path = Path(destination_path)

    result = {
        "status": "unknown",
        "operation_used": None,
        "error_type": None,
        "error_message": None,
    }

    try:
        if not source_path.exists():
            raise FileNotFoundError(f"Source file does not exist: {source_path}")

        if not source_path.is_file():
            raise ValueError(f"Source path is not a regular file: {source_path}")

        destination_path.parent.mkdir(parents=True, exist_ok=True)

        if destination_path.exists() or destination_path.is_symlink():
            if on_existing == "error":
                raise FileExistsError(f"Destination already exists: {destination_path}")
            if on_existing == "skip":
                result["status"] = "skipped_existing"
                result["operation_used"] = "skip"
                return result
            if on_existing == "overwrite":
                remove_existing_file(destination_path)

        if mode == "copy":
            shutil.copy2(source_path, destination_path)
            result["status"] = "ok"
            result["operation_used"] = "copy"
            return result

        if mode == "symlink":
            destination_path.symlink_to(source_path)
            result["status"] = "ok"
            result["operation_used"] = "symlink"
            return result

        if mode == "hardlink":
            try:
                os.link(source_path, destination_path)
                result["status"] = "ok"
                result["operation_used"] = "hardlink"
                return result
            except OSError as hardlink_exc:
                if not allow_hardlink_fallback_to_copy:
                    raise

                shutil.copy2(source_path, destination_path)
                result["status"] = "ok"
                result["operation_used"] = "copy_fallback_from_hardlink"
                result["error_type"] = type(hardlink_exc).__name__
                result["error_message"] = str(hardlink_exc)
                return result

        raise ValueError(f"Unsupported materialization mode: {mode}")

    except Exception as exc:
        result["status"] = "error"
        result["operation_used"] = mode
        result["error_type"] = type(exc).__name__
        result["error_message"] = str(exc)
        return result


print("Helper functions loaded.")

Helper functions loaded.


# Validate clustering manifest and prepare clusters to name


In [6]:
required_cluster_columns = {
    "embedding_row",
    "cluster_label",
    "cluster_folder_stub",
    "crop_path",
    "source_path",
    "relative_path",
    "cluster_probability",
    "outlier_score",
    "requires_manual_review",
}

missing_cluster_columns = sorted(required_cluster_columns.difference(set(cluster_manifest_df.columns)))

if missing_cluster_columns:
    raise ValueError(f"cluster_manifest_df is missing required columns: {missing_cluster_columns}")

cluster_work_df = cluster_manifest_df.copy()

cluster_work_df["crop_path"] = cluster_work_df["crop_path"].map(normalize_path_string)
cluster_work_df["source_path"] = cluster_work_df["source_path"].map(normalize_path_string)
cluster_work_df["requires_manual_review"] = cluster_work_df["requires_manual_review"].map(safe_bool)
cluster_work_df["cluster_label"] = cluster_work_df["cluster_label"].astype(int)

cluster_work_df["crop_exists"] = cluster_work_df["crop_path"].map(lambda p: Path(p).exists())
cluster_work_df["source_exists"] = cluster_work_df["source_path"].map(lambda p: Path(p).exists())

if TAG_ONLY_NON_NOISE_CLUSTERS:
    cluster_work_df = cluster_work_df[cluster_work_df["cluster_label"].ne(-1)].copy()

if IGNORE_REVIEW_FOLDERS_FOR_NAMING:
    cluster_work_df = cluster_work_df[~cluster_work_df["requires_manual_review"]].copy()

cluster_sizes = cluster_work_df.groupby("cluster_label").size().reset_index(name="cluster_size")

eligible_labels = cluster_sizes[
    cluster_sizes["cluster_size"].ge(MIN_CLUSTER_SIZE_TO_NAME)
]["cluster_label"].astype(int).tolist()

cluster_work_df = cluster_work_df[cluster_work_df["cluster_label"].isin(eligible_labels)].copy()

if cluster_work_df.empty:
    raise RuntimeError("No clusters are eligible for semantic naming under the current configuration.")

print(f"Eligible clusters: {len(eligible_labels)}")
print(f"Eligible image rows: {len(cluster_work_df)}")
print(f"Missing crop files among eligible rows: {int((~cluster_work_df['crop_exists']).sum())}")

Eligible clusters: 188
Eligible image rows: 6567
Missing crop files among eligible rows: 0


# Select representative images per cluster for tagging

In [7]:
sample_rows = []

for cluster_label, group in cluster_work_df.groupby("cluster_label", dropna=False):
    group = group[group["crop_exists"]].copy()

    if group.empty:
        continue

    selected = select_cluster_samples(group, max_images=MAX_IMAGES_PER_CLUSTER_TO_TAG)
    selected["selected_for_tagging"] = True
    sample_rows.append(selected)

if not sample_rows:
    raise RuntimeError("No valid crop files were selected for tagging.")

tagging_input_df = pd.concat(sample_rows, axis=0).reset_index(drop=True)
tagging_input_df.insert(0, "tagging_index", np.arange(len(tagging_input_df), dtype=int))

tagging_input_path = TABLES_DIR / "tagging_input_manifest.csv"
tagging_input_df.to_csv(tagging_input_path, index=False)

print(f"Images selected for tagging: {len(tagging_input_df)}")
print(f"Clusters represented in tagging input: {tagging_input_df['cluster_label'].nunique()}")
tagging_input_df[[
    "tagging_index",
    "cluster_label",
    "cluster_folder_stub",
    "relative_path",
    "cluster_probability",
    "outlier_score",
    "crop_path",
]].head(20)

Images selected for tagging: 1893
Clusters represented in tagging input: 188


,tagging_index,cluster_label,cluster_folder_stub,relative_path,cluster_probability,outlier_score,crop_path
0,0,0,cluster_00000_unknown,IMG_6155.JPG,1.000000,0.000000,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
1,1,0,cluster_00000_unknown,IMG_5836.JPG,1.000000,0.000000,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
2,2,0,cluster_00000_unknown,IMG_3982.JPG,1.000000,0.000000,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
3,3,0,cluster_00000_unknown,20260506_143232.jpg,1.000000,0.000000,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
4,4,0,cluster_00000_unknown,IMG_3221.JPG,1.000000,0.000000,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
5,5,0,cluster_00000_unknown,IMG_1108.JPG,0.979539,0.020461,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
6,6,1,cluster_00001_unknown,20251111_181504.jpg,1.000000,0.000000,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
7,7,1,cluster_00001_unknown,20260403_091948.jpg,1.000000,0.000000,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
8,8,1,cluster_00001_unknown,IMG_0605.JPG,1.000000,0.000000,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
9,9,1,cluster_00001_unknown,20251228_224203.jpg,1.000000,0.000000,/home/ssparzival/Proyectos/GitHub/Automatic-Or...


# Validate selected crop files

In [9]:
validation_rows = []

for _, row in tqdm(tagging_input_df.iterrows(), total=len(tagging_input_df), desc="Validating tagging input images"):
    is_valid, error_type, error_message = validate_image_file(row["crop_path"])

    validation_rows.append({
        "tagging_index": int(row["tagging_index"]),
        "cluster_label": int(row["cluster_label"]),
        "crop_path": row["crop_path"],
        "valid_for_tagging": bool(is_valid),
        "validation_error_type": error_type,
        "validation_error_message": error_message,
    })

tagging_validation_df = pd.DataFrame(validation_rows)

tagging_ready_df = tagging_input_df.merge(
    tagging_validation_df,
    on=["tagging_index", "cluster_label", "crop_path"],
    how="left",
    validate="one_to_one",
)

tagging_ready_df = tagging_ready_df[tagging_ready_df["valid_for_tagging"].astype(bool)].copy()
tagging_ready_df = tagging_ready_df.reset_index(drop=True)

invalid_tagging_input_df = tagging_validation_df[
    ~tagging_validation_df["valid_for_tagging"].astype(bool)
].copy()

tagging_validation_path = TABLES_DIR / "tagging_validation.csv"
invalid_tagging_input_path = TABLES_DIR / "invalid_tagging_input.csv"
tagging_ready_path = TABLES_DIR / "tagging_ready_manifest.csv"

tagging_validation_df.to_csv(tagging_validation_path, index=False)
invalid_tagging_input_df.to_csv(invalid_tagging_input_path, index=False)
tagging_ready_df.to_csv(tagging_ready_path, index=False)

print(f"Valid images for tagging: {len(tagging_ready_df)}")
print(f"Invalid images for tagging: {len(invalid_tagging_input_df)}")

if tagging_ready_df.empty:
    raise RuntimeError("No valid images are available for anime tagging.")


Validating tagging input images:   0%|          | 0/1893 [00:00<?, ?it/s]

Valid images for tagging: 1893
Invalid images for tagging: 0


# Warm up tagger

In [10]:
warmup_path = tagging_ready_df.iloc[0]["crop_path"]

try:
    warmup_tags, warmup_fallback_error_type, warmup_fallback_error_message = extract_tags_with_fallback(warmup_path)
except Exception as exc:
    raise RuntimeError("Tagger warmup failed unexpectedly.") from exc

if warmup_tags is None:
    raise RuntimeError(
        "Tagger warmup failed. "
        f"Error type: {warmup_fallback_error_type}. "
        f"Error message: {warmup_fallback_error_message}"
    )

print("Tagger warmup completed.")
print(f"Tagger used: {warmup_tags.get('tagger')}")
print(f"Character tags found in warmup: {len(warmup_tags.get('character_tags', {}))}")
print(f"General tags found in warmup: {len(warmup_tags.get('general_tags', {}))}")

selected_tags.csv:   0%|          | 0.00/597k [00:00<?, ?B/s]

model.onnx:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

preprocess.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

thresholds.csv:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

Tagger warmup completed.
Tagger used: pixai
Character tags found in warmup: 0
General tags found in warmup: 43


# Run anime tagging

In [11]:
tagging_rows = []

for _, row in tqdm(tagging_ready_df.iterrows(), total=len(tagging_ready_df), desc="Tagging representative crop images"):
    tags, fallback_error_type, fallback_error_message = extract_tags_with_fallback(row["crop_path"])

    if tags is None:
        tagging_rows.append({
            "tagging_index": int(row["tagging_index"]),
            "embedding_row": int(row["embedding_row"]),
            "cluster_label": int(row["cluster_label"]),
            "cluster_folder_stub": row["cluster_folder_stub"],
            "relative_path": row["relative_path"],
            "crop_path": row["crop_path"],
            "source_path": row["source_path"],
            "tagging_status": "error",
            "tagger_used": None,
            "error_type": fallback_error_type,
            "error_message": fallback_error_message,
            "pixai_fallback_error_type": fallback_error_type,
            "pixai_fallback_error_message": fallback_error_message,
            "character_tags_json": "{}",
            "general_tags_json": "{}",
            "ips_json": "[]",
            "ips_mapping_json": "{}",
            "top_character_tag": None,
            "top_character_score": np.nan,
            "top_general_tag": None,
            "top_general_score": np.nan,
            "processed_at": now_iso(),
        })
        continue

    character_tags = tags.get("character_tags", {}) or {}
    general_tags = tags.get("general_tags", {}) or {}
    ips = tags.get("ips", []) or []
    ips_mapping = tags.get("ips_mapping", {}) or {}

    character_tags = {
        normalize_tag_name(k): float(v)
        for k, v in character_tags.items()
        if float(v) >= MIN_CHARACTER_SCORE
    }

    top_character_tag = None
    top_character_score = np.nan

    if character_tags:
        top_character_tag, top_character_score = max(character_tags.items(), key=lambda item: item[1])

    top_general_tag = None
    top_general_score = np.nan

    if general_tags:
        top_general_tag, top_general_score = max(general_tags.items(), key=lambda item: item[1])

    tagging_rows.append({
        "tagging_index": int(row["tagging_index"]),
        "embedding_row": int(row["embedding_row"]),
        "cluster_label": int(row["cluster_label"]),
        "cluster_folder_stub": row["cluster_folder_stub"],
        "relative_path": row["relative_path"],
        "crop_path": row["crop_path"],
        "source_path": row["source_path"],
        "tagging_status": "ok",
        "tagger_used": tags.get("tagger"),
        "error_type": None,
        "error_message": None,
        "pixai_fallback_error_type": fallback_error_type,
        "pixai_fallback_error_message": fallback_error_message,
        "character_tags_json": safe_json_dumps(character_tags),
        "general_tags_json": safe_json_dumps(general_tags),
        "ips_json": json.dumps(ips, ensure_ascii=False),
        "ips_mapping_json": safe_json_dumps(ips_mapping),
        "top_character_tag": top_character_tag,
        "top_character_score": float(top_character_score) if pd.notna(top_character_score) else np.nan,
        "top_general_tag": top_general_tag,
        "top_general_score": float(top_general_score) if pd.notna(top_general_score) else np.nan,
        "processed_at": now_iso(),
    })

    gc.collect()

tagging_manifest_df = pd.DataFrame(tagging_rows)

tagging_manifest_path = TABLES_DIR / "tagging_manifest.csv"
tagging_manifest_df.to_csv(tagging_manifest_path, index=False)

print(f"Tagging rows: {len(tagging_manifest_df)}")
print(f"Successful taggings: {int(tagging_manifest_df['tagging_status'].eq('ok').sum())}")
print(f"Tagging errors: {int(tagging_manifest_df['tagging_status'].ne('ok').sum())}")

Tagging representative crop images:   0%|          | 0/1893 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Aggregate character tags by cluster

In [ ]:
cluster_name_rows = []

for cluster_label, group in tagging_manifest_df.groupby("cluster_label", dropna=False):
    cluster_label = int(cluster_label)

    ok_group = group[group["tagging_status"].eq("ok")].copy()
    cluster_folder_stub = str(group["cluster_folder_stub"].dropna().iloc[0]) if group["cluster_folder_stub"].notna().any() else f"cluster_{cluster_label:05d}_unknown"

    tag_weight = defaultdict(float)
    tag_count = defaultdict(int)
    tag_scores = defaultdict(list)
    tag_examples = defaultdict(list)

    for _, row in ok_group.iterrows():
        character_tags = parse_dict_like(row["character_tags_json"])

        for tag, score in character_tags.items():
            score = float(score)
            tag = normalize_tag_name(tag)

            if score < MIN_CHARACTER_SCORE:
                continue

            tag_weight[tag] += score
            tag_count[tag] += 1
            tag_scores[tag].append(score)

            if len(tag_examples[tag]) < 5:
                tag_examples[tag].append(row["relative_path"])

    total_tagged_images = int(len(ok_group))
    unique_candidate_tags = len(tag_weight)

    if tag_weight:
        ranked_tags = sorted(
            tag_weight.keys(),
            key=lambda tag: (
                tag_weight[tag] / max(1, total_tagged_images),
                tag_count[tag],
                np.mean(tag_scores[tag]),
                tag,
            ),
            reverse=True,
        )

        best_tag = ranked_tags[0]
        second_tag = ranked_tags[1] if len(ranked_tags) > 1 else None

        best_count = int(tag_count[best_tag])
        best_share = float(best_count / max(1, total_tagged_images))
        best_weighted_score = float(tag_weight[best_tag] / max(1, total_tagged_images))
        best_mean_score = float(np.mean(tag_scores[best_tag]))
        best_max_score = float(np.max(tag_scores[best_tag]))

        if second_tag is not None:
            second_weighted_score = float(tag_weight[second_tag] / max(1, total_tagged_images))
            second_count = int(tag_count[second_tag])
            second_share = float(second_count / max(1, total_tagged_images))
        else:
            second_weighted_score = 0.0
            second_count = 0
            second_share = 0.0

        margin = float(best_weighted_score - second_weighted_score)

        accepted = (
            best_share >= MIN_NAME_SHARE and
            best_weighted_score >= MIN_WEIGHTED_SCORE and
            margin >= MIN_TOP_MARGIN
        )

        suggested_name = display_tag_name(best_tag) if accepted else None

        top_tags_payload = []
        for tag in ranked_tags[:10]:
            top_tags_payload.append({
                "tag": tag,
                "count": int(tag_count[tag]),
                "share": float(tag_count[tag] / max(1, total_tagged_images)),
                "weighted_score": float(tag_weight[tag] / max(1, total_tagged_images)),
                "mean_score": float(np.mean(tag_scores[tag])),
                "max_score": float(np.max(tag_scores[tag])),
                "examples": tag_examples[tag],
            })

    else:
        best_tag = None
        second_tag = None
        best_count = 0
        best_share = 0.0
        best_weighted_score = 0.0
        best_mean_score = np.nan
        best_max_score = np.nan
        second_weighted_score = 0.0
        second_count = 0
        second_share = 0.0
        margin = 0.0
        accepted = False
        suggested_name = None
        top_tags_payload = []

    proposed_named_folder = (
        sanitize_filename_component(suggested_name, fallback=cluster_folder_stub)
        if suggested_name is not None
        else cluster_folder_stub
    )

    if suggested_name is not None:
        proposed_named_folder = f"{proposed_named_folder}__{cluster_folder_stub}"

    cluster_name_rows.append({
        "cluster_label": cluster_label,
        "cluster_folder_stub": cluster_folder_stub,
        "tagged_image_count": total_tagged_images,
        "tagging_error_count": int(group["tagging_status"].ne("ok").sum()),
        "unique_candidate_character_tags": int(unique_candidate_tags),
        "accepted_name": bool(accepted),
        "suggested_character_tag": suggested_name,
        "proposed_named_folder": sanitize_filename_component(proposed_named_folder, fallback=cluster_folder_stub),
        "best_raw_tag": best_tag,
        "best_tag_count": int(best_count),
        "best_tag_share": float(best_share),
        "best_weighted_score": float(best_weighted_score),
        "best_mean_score": float(best_mean_score) if pd.notna(best_mean_score) else np.nan,
        "best_max_score": float(best_max_score) if pd.notna(best_max_score) else np.nan,
        "second_raw_tag": second_tag,
        "second_tag_count": int(second_count),
        "second_tag_share": float(second_share),
        "second_weighted_score": float(second_weighted_score),
        "top_margin": float(margin),
        "top_tags_json": safe_json_dumps(top_tags_payload),
        "naming_rule": (
            f"accepted if share >= {MIN_NAME_SHARE}, "
            f"weighted_score >= {MIN_WEIGHTED_SCORE}, "
            f"margin >= {MIN_TOP_MARGIN}"
        ),
    })

cluster_name_suggestions_df = pd.DataFrame(cluster_name_rows)

cluster_name_suggestions_df = cluster_name_suggestions_df.sort_values(
    ["accepted_name", "best_weighted_score", "best_tag_share", "tagged_image_count"],
    ascending=[False, False, False, False],
).reset_index(drop=True)

cluster_name_suggestions_path = TABLES_DIR / "cluster_name_suggestions.csv"
cluster_name_suggestions_df.to_csv(cluster_name_suggestions_path, index=False)

print(f"Cluster name suggestions: {len(cluster_name_suggestions_df)}")
print(f"Accepted semantic names: {int(cluster_name_suggestions_df['accepted_name'].sum())}")

cluster_name_suggestions_df.head(30)

# Create folder rename plan

In [ ]:
if CREATE_RENAME_PLAN:
    folder_rename_plan_df = folder_assignment_df.copy()

    folder_rename_plan_df["cluster_label"] = folder_rename_plan_df["cluster_label"].astype(int)

    name_lookup_df = cluster_name_suggestions_df[[
        "cluster_label",
        "cluster_folder_stub",
        "accepted_name",
        "suggested_character_tag",
        "proposed_named_folder",
        "best_tag_share",
        "best_weighted_score",
        "top_margin",
        "top_tags_json",
    ]].copy()

    folder_rename_plan_df = folder_rename_plan_df.merge(
        name_lookup_df,
        on="cluster_label",
        how="left",
        suffixes=("", "_naming"),
    )

    folder_rename_plan_df["accepted_name"] = folder_rename_plan_df["accepted_name"].fillna(False).map(safe_bool)

    folder_rename_plan_df["named_folder"] = folder_rename_plan_df["proposed_folder"]

    accepted_mask = (
        folder_rename_plan_df["accepted_name"] &
        folder_rename_plan_df["proposed_named_folder"].notna() &
        folder_rename_plan_df["cluster_label"].ne(-1)
    )

    folder_rename_plan_df.loc[accepted_mask, "named_folder"] = folder_rename_plan_df.loc[
        accepted_mask,
        "proposed_named_folder",
    ]

    folder_rename_plan_df["named_folder"] = folder_rename_plan_df["named_folder"].map(
        lambda value: sanitize_filename_component(value, fallback="_needs_review")
    )

    folder_rename_plan_path = TABLES_DIR / "folder_rename_plan.csv"
    folder_rename_plan_df.to_csv(folder_rename_plan_path, index=False)

    folder_rename_preview_df = (
        folder_rename_plan_df
        .groupby(["proposed_folder", "named_folder", "accepted_name", "suggested_character_tag"], dropna=False)
        .size()
        .reset_index(name="image_count")
        .sort_values(["accepted_name", "image_count"], ascending=[False, False])
        .reset_index(drop=True)
    )

    folder_rename_preview_path = TABLES_DIR / "folder_rename_preview.csv"
    folder_rename_preview_df.to_csv(folder_rename_preview_path, index=False)

else:
    folder_rename_plan_df = pd.DataFrame()
    folder_rename_preview_df = pd.DataFrame()
    folder_rename_plan_path = None
    folder_rename_preview_path = None

folder_rename_preview_df.head(40)

# Write cluster naming metadata files

In [ ]:
CLUSTER_NAMING_METADATA_DIR = RUN_DIR / "cluster_naming_metadata"
CLUSTER_NAMING_METADATA_DIR.mkdir(parents=True, exist_ok=True)

metadata_rows = []

for _, row in cluster_name_suggestions_df.iterrows():
    cluster_label = int(row["cluster_label"])
    cluster_folder_stub = row["cluster_folder_stub"]

    cluster_tagging_df = tagging_manifest_df[
        tagging_manifest_df["cluster_label"].eq(cluster_label)
    ].copy()

    metadata = {
        "cluster_label": cluster_label,
        "cluster_folder_stub": cluster_folder_stub,
        "accepted_name": bool(row["accepted_name"]),
        "suggested_character_tag": None if pd.isna(row["suggested_character_tag"]) else row["suggested_character_tag"],
        "proposed_named_folder": row["proposed_named_folder"],
        "tagged_image_count": int(row["tagged_image_count"]),
        "tagging_error_count": int(row["tagging_error_count"]),
        "unique_candidate_character_tags": int(row["unique_candidate_character_tags"]),
        "best_raw_tag": None if pd.isna(row["best_raw_tag"]) else row["best_raw_tag"],
        "best_tag_count": int(row["best_tag_count"]),
        "best_tag_share": float(row["best_tag_share"]),
        "best_weighted_score": float(row["best_weighted_score"]),
        "top_margin": float(row["top_margin"]),
        "top_tags": json.loads(row["top_tags_json"]) if pd.notna(row["top_tags_json"]) else [],
        "rules": {
            "min_character_score": float(MIN_CHARACTER_SCORE),
            "min_name_share": float(MIN_NAME_SHARE),
            "min_weighted_score": float(MIN_WEIGHTED_SCORE),
            "min_top_margin": float(MIN_TOP_MARGIN),
            "max_images_per_cluster_to_tag": int(MAX_IMAGES_PER_CLUSTER_TO_TAG),
        },
        "tagger": {
            "primary": PRIMARY_TAGGER,
            "pixai_model_name": PIXAI_MODEL_NAME,
            "wd14_fallback_enabled": bool(USE_WD14_FALLBACK),
            "wd14_available": bool(WD14_AVAILABLE),
            "wd14_model_name": WD14_MODEL_NAME,
        },
        "sample_tagged_items": cluster_tagging_df[[
            "tagging_index",
            "embedding_row",
            "relative_path",
            "crop_path",
            "tagging_status",
            "tagger_used",
            "top_character_tag",
            "top_character_score",
            "top_general_tag",
            "top_general_score",
        ]].replace({np.nan: None}).head(25).to_dict(orient="records"),
        "created_at": now_iso(),
    }

    metadata_path = CLUSTER_NAMING_METADATA_DIR / f"{sanitize_filename_component(cluster_folder_stub)}.json"

    with metadata_path.open("w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    metadata_rows.append({
        "cluster_label": cluster_label,
        "cluster_folder_stub": cluster_folder_stub,
        "metadata_path": metadata_path.as_posix(),
    })

cluster_naming_metadata_index_df = pd.DataFrame(metadata_rows)
cluster_naming_metadata_index_path = TABLES_DIR / "cluster_naming_metadata_index.csv"
cluster_naming_metadata_index_df.to_csv(cluster_naming_metadata_index_path, index=False)

cluster_naming_metadata_index_df.head(20)

# Create naming contact sheets

In [ ]:
contact_sheet_rows = []

sheet_clusters_df = cluster_name_suggestions_df.sort_values(
    ["accepted_name", "best_weighted_score", "tagged_image_count"],
    ascending=[False, False, False],
).head(MAX_CONTACT_SHEETS)

for _, row in tqdm(
    sheet_clusters_df.iterrows(),
    total=len(sheet_clusters_df),
    desc="Creating naming contact sheets",
):
    cluster_label = int(row["cluster_label"])
    cluster_folder_stub = row["cluster_folder_stub"]
    proposed_name = row["proposed_named_folder"]

    group = cluster_work_df[cluster_work_df["cluster_label"].eq(cluster_label)].copy()
    group = select_cluster_samples(group, max_images=CONTACT_SHEET_MAX_IMAGES_PER_CLUSTER)

    output_name = sanitize_filename_component(proposed_name, fallback=cluster_folder_stub)
    output_path = CONTACT_SHEETS_DIR / f"{output_name}.jpg"

    title = (
        f"{proposed_name} | accepted={bool(row['accepted_name'])} | "
        f"score={row['best_weighted_score']:.3f}"
    )

    created = create_contact_sheet(
        image_paths=group["crop_path"].dropna().tolist(),
        output_path=output_path,
        thumb_size=CONTACT_SHEET_THUMB_SIZE,
        columns=CONTACT_SHEET_COLUMNS,
        title=title,
    )

    contact_sheet_rows.append({
        "cluster_label": cluster_label,
        "cluster_folder_stub": cluster_folder_stub,
        "proposed_named_folder": proposed_name,
        "accepted_name": bool(row["accepted_name"]),
        "image_count": int(len(group)),
        "created": bool(created),
        "contact_sheet_path": output_path.as_posix() if created else None,
    })

naming_contact_sheets_df = pd.DataFrame(contact_sheet_rows)
naming_contact_sheets_path = TABLES_DIR / "naming_contact_sheets.csv"
naming_contact_sheets_df.to_csv(naming_contact_sheets_path, index=False)

naming_contact_sheets_df.head(20)

# Optionally create non-destructive named output


In [ ]:
named_materialization_rows = []

if CREATE_NAMED_OUTPUT:
    if folder_rename_plan_df.empty:
        raise RuntimeError("CREATE_NAMED_OUTPUT=True requires a non-empty folder_rename_plan_df.")

    named_plan_df = folder_rename_plan_df.copy()

    if "source_path" not in named_plan_df.columns:
        raise ValueError("folder_rename_plan_df must contain source_path.")

    named_plan_df["source_path"] = named_plan_df["source_path"].map(normalize_path_string)
    named_plan_df["named_folder"] = named_plan_df["named_folder"].map(
        lambda value: sanitize_filename_component(value, fallback="_needs_review")
    )

    used_destinations = set()
    destination_rows = []

    for _, row in named_plan_df.iterrows():
        source_path = Path(row["source_path"])
        named_folder = row["named_folder"]

        try:
            embedding_row = int(row["embedding_row"])
            prefix = f"{embedding_row:07d}"
        except Exception:
            prefix = stable_short_hash(source_path.as_posix(), length=10)

        destination_filename = f"{prefix}__{sanitize_filename_component(source_path.stem, fallback='image')}{source_path.suffix}"
        destination_path = NAMED_OUTPUT_DIR / named_folder / destination_filename

        destination_key = destination_path.as_posix()

        counter = 1

        while destination_key in used_destinations or destination_path.exists():
            destination_filename = f"{prefix}__{sanitize_filename_component(source_path.stem, fallback='image')}__dup{counter:03d}{source_path.suffix}"
            destination_path = NAMED_OUTPUT_DIR / named_folder / destination_filename
            destination_key = destination_path.as_posix()
            counter += 1

        used_destinations.add(destination_key)

        destination_rows.append({
            "embedding_row": int(row["embedding_row"]),
            "source_path": source_path.as_posix(),
            "named_folder": named_folder,
            "named_destination_path": destination_path.as_posix(),
        })

    named_destination_df = pd.DataFrame(destination_rows)

    named_plan_df = named_plan_df.merge(
        named_destination_df,
        on=["embedding_row", "source_path", "named_folder"],
        how="left",
        validate="one_to_one",
    )

    named_output_plan_path = TABLES_DIR / "named_output_plan.csv"
    named_plan_df.to_csv(named_output_plan_path, index=False)

    for _, row in tqdm(named_plan_df.iterrows(), total=len(named_plan_df), desc="Creating named output"):
        result = materialize_one_file(
            source_path=Path(row["source_path"]),
            destination_path=Path(row["named_destination_path"]),
            mode=NAMED_OUTPUT_MODE,
            on_existing=ON_EXISTING,
            allow_hardlink_fallback_to_copy=ALLOW_HARDLINK_FALLBACK_TO_COPY,
        )

        named_materialization_rows.append({
            "embedding_row": int(row["embedding_row"]),
            "source_path": row["source_path"],
            "named_destination_path": row["named_destination_path"],
            "named_folder": row["named_folder"],
            "cluster_label": int(row["cluster_label"]),
            "accepted_name": safe_bool(row["accepted_name"]),
            "suggested_character_tag": row.get("suggested_character_tag"),
            "status": result["status"],
            "operation_used": result["operation_used"],
            "error_type": result["error_type"],
            "error_message": result["error_message"],
            "processed_at": now_iso(),
        })

    named_materialization_df = pd.DataFrame(named_materialization_rows)

else:
    named_output_plan_path = None
    named_materialization_df = pd.DataFrame()

named_materialization_path = TABLES_DIR / "named_materialization_result.csv"
named_materialization_df.to_csv(named_materialization_path, index=False)

named_materialization_df.head(20)

# Write output README for naming run


In [ ]:
readme_path = RUN_DIR / "README.md"

accepted_count = int(cluster_name_suggestions_df["accepted_name"].sum())
total_suggestions = int(len(cluster_name_suggestions_df))

tagger_counts_df = (
    tagging_manifest_df
    .groupby(["tagging_status", "tagger_used"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

readme_lines = [
    "# Cluster Naming Run",
    "",
    "This run uses anime image taggers to suggest semantic names for visually discovered clusters.",
    "",
    "## Important",
    "",
    "- The visual clusters remain the source of truth.",
    "- Tagger names are suggestions, not guaranteed labels.",
    "- A suggested name is accepted only if it passes the configured agreement thresholds.",
    "",
    "## Configuration",
    "",
    f"- Primary tagger: `{PRIMARY_TAGGER}`",
    f"- PixAI model: `{PIXAI_MODEL_NAME}`",
    f"- WD14 fallback enabled: `{USE_WD14_FALLBACK}`",
    f"- WD14 available: `{WD14_AVAILABLE}`",
    f"- WD14 model: `{WD14_MODEL_NAME}`",
    f"- Minimum character score: `{MIN_CHARACTER_SCORE}`",
    f"- Minimum name share: `{MIN_NAME_SHARE}`",
    f"- Minimum weighted score: `{MIN_WEIGHTED_SCORE}`",
    f"- Minimum top margin: `{MIN_TOP_MARGIN}`",
    f"- Max images per cluster tagged: `{MAX_IMAGES_PER_CLUSTER_TO_TAG}`",
    "",
    "## Results",
    "",
    f"- Clusters with naming suggestions: `{total_suggestions}`",
    f"- Accepted semantic names: `{accepted_count}`",
    f"- Tagging rows: `{len(tagging_manifest_df)}`",
    "",
    "## Tagger usage counts",
    "",
    tagger_counts_df.to_markdown(index=False),
    "",
    "## Key files",
    "",
    f"- `tables/tagging_manifest.csv`",
    f"- `tables/cluster_name_suggestions.csv`",
    f"- `tables/folder_rename_plan.csv`",
    f"- `tables/folder_rename_preview.csv`",
    f"- `contact_sheets/`",
    "",
]

with readme_path.open("w", encoding="utf-8") as f:
    f.write("\n".join(readme_lines))

print(f"README written to: {readme_path}")

# Consistency checks

In [ ]:
checks = {
    "tagging_input_exists": tagging_input_path.exists(),
    "tagging_ready_exists": tagging_ready_path.exists(),
    "tagging_manifest_exists": tagging_manifest_path.exists(),
    "cluster_name_suggestions_exists": cluster_name_suggestions_path.exists(),
    "cluster_naming_metadata_index_exists": cluster_naming_metadata_index_path.exists(),
    "naming_contact_sheets_exists": naming_contact_sheets_path.exists(),
    "readme_exists": readme_path.exists(),
    "tagging_manifest_rows_match_ready_or_less": bool(len(tagging_manifest_df) <= len(tagging_ready_df)),
    "cluster_suggestions_non_empty": bool(len(cluster_name_suggestions_df) > 0),
    "no_duplicate_cluster_suggestions": bool(not cluster_name_suggestions_df["cluster_label"].duplicated().any()),
}

if CREATE_RENAME_PLAN:
    checks["folder_rename_plan_exists"] = folder_rename_plan_path.exists()
    checks["folder_rename_preview_exists"] = folder_rename_preview_path.exists()
    checks["folder_rename_plan_rows_match_assignment"] = bool(len(folder_rename_plan_df) == len(folder_assignment_df))

if CREATE_NAMED_OUTPUT:
    checks["named_output_dir_exists"] = NAMED_OUTPUT_DIR.exists()
    checks["named_output_plan_exists"] = named_output_plan_path.exists()
    checks["named_materialization_result_exists"] = named_materialization_path.exists()
    checks["no_named_materialization_errors"] = bool(
        named_materialization_df.empty or not named_materialization_df["status"].eq("error").any()
    )

failed_checks = {key: value for key, value in checks.items() if not value}

checks_path = REPORTS_DIR / "consistency_checks.json"

with checks_path.open("w", encoding="utf-8") as f:
    json.dump(checks, f, ensure_ascii=False, indent=2)

if failed_checks:
    raise RuntimeError(f"Consistency checks failed: {failed_checks}")

checks

# Save run summary

In [ ]:
summary = {
    "run_id": RUN_ID,
    "created_at": now_iso(),
    "project_dir": PROJECT_DIR.as_posix(),
    "run_dir": RUN_DIR.as_posix(),
    "previous_clustering_run_dir": PREVIOUS_CLUSTERING_RUN_DIR.as_posix(),
    "previous_materialization_run_dir": PREVIOUS_MATERIALIZATION_RUN_DIR.as_posix() if PREVIOUS_MATERIALIZATION_RUN_DIR is not None else None,
    "primary_tagger": PRIMARY_TAGGER,
    "pixai_model_name": PIXAI_MODEL_NAME,
    "use_wd14_fallback": bool(USE_WD14_FALLBACK),
    "wd14_available": bool(WD14_AVAILABLE),
    "wd14_model_name": WD14_MODEL_NAME,
    "min_cluster_size_to_name": int(MIN_CLUSTER_SIZE_TO_NAME),
    "max_images_per_cluster_to_tag": int(MAX_IMAGES_PER_CLUSTER_TO_TAG),
    "min_character_score": float(MIN_CHARACTER_SCORE),
    "min_name_share": float(MIN_NAME_SHARE),
    "min_weighted_score": float(MIN_WEIGHTED_SCORE),
    "min_top_margin": float(MIN_TOP_MARGIN),
    "eligible_clusters": int(len(eligible_labels)),
    "tagging_input_images": int(len(tagging_input_df)),
    "tagging_ready_images": int(len(tagging_ready_df)),
    "successful_taggings": int(tagging_manifest_df["tagging_status"].eq("ok").sum()),
    "tagging_errors": int(tagging_manifest_df["tagging_status"].ne("ok").sum()),
    "cluster_name_suggestions": int(len(cluster_name_suggestions_df)),
    "accepted_semantic_names": int(cluster_name_suggestions_df["accepted_name"].sum()),
    "create_named_output": bool(CREATE_NAMED_OUTPUT),
    "named_output_dir": NAMED_OUTPUT_DIR.as_posix() if CREATE_NAMED_OUTPUT else None,
    "python": sys.version,
    "platform": platform.platform(),
    "outputs": {
        "tagging_input_manifest": tagging_input_path.as_posix(),
        "tagging_validation": tagging_validation_path.as_posix(),
        "invalid_tagging_input": invalid_tagging_input_path.as_posix(),
        "tagging_ready_manifest": tagging_ready_path.as_posix(),
        "tagging_manifest": tagging_manifest_path.as_posix(),
        "cluster_name_suggestions": cluster_name_suggestions_path.as_posix(),
        "cluster_naming_metadata_index": cluster_naming_metadata_index_path.as_posix(),
        "naming_contact_sheets": naming_contact_sheets_path.as_posix(),
        "readme": readme_path.as_posix(),
        "consistency_checks": checks_path.as_posix(),
    },
}

if CREATE_RENAME_PLAN:
    summary["outputs"]["folder_rename_plan"] = folder_rename_plan_path.as_posix()
    summary["outputs"]["folder_rename_preview"] = folder_rename_preview_path.as_posix()

if CREATE_NAMED_OUTPUT:
    summary["outputs"]["named_output_plan"] = named_output_plan_path.as_posix()
    summary["outputs"]["named_materialization_result"] = named_materialization_path.as_posix()
    summary["outputs"]["named_output_dir"] = NAMED_OUTPUT_DIR.as_posix()

summary_path = REPORTS_DIR / "summary.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved outputs:")
for key, value in summary["outputs"].items():
    print(f"{key}: {value}")

print(f"summary: {summary_path.as_posix()}")

# Inspect accepted names


In [ ]:
accepted_names_df = cluster_name_suggestions_df[
    cluster_name_suggestions_df["accepted_name"]
].copy()

if accepted_names_df.empty:
    print("No semantic names were accepted under the current thresholds.")
else:
    accepted_names_df[[
        "cluster_label",
        "cluster_folder_stub",
        "suggested_character_tag",
        "proposed_named_folder",
        "tagged_image_count",
        "best_tag_count",
        "best_tag_share",
        "best_weighted_score",
        "top_margin",
    ]].head(80)

# Inspect uncertain naming cases


In [ ]:
uncertain_names_df = cluster_name_suggestions_df[
    ~cluster_name_suggestions_df["accepted_name"]
].copy()

if uncertain_names_df.empty:
    print("All cluster naming suggestions were accepted.")
else:
    uncertain_names_df[[
        "cluster_label",
        "cluster_folder_stub",
        "best_raw_tag",
        "best_tag_count",
        "best_tag_share",
        "best_weighted_score",
        "second_raw_tag",
        "second_weighted_score",
        "top_margin",
        "tagged_image_count",
        "unique_candidate_character_tags",
    ]].head(80)

# Inspect tagger errors, if any


In [ ]:
tagging_errors_df = tagging_manifest_df[
    tagging_manifest_df["tagging_status"].ne("ok")
].copy()

if tagging_errors_df.empty:
    print("No tagging errors detected.")
else:
    tagging_errors_df[[
        "tagging_index",
        "cluster_label",
        "relative_path",
        "crop_path",
        "error_type",
        "error_message",
    ]].head(100)

# Inspect rename preview

In [ ]:
if CREATE_RENAME_PLAN:
    folder_rename_preview_df.head(100)
else:
    print("CREATE_RENAME_PLAN is disabled.")

# Final output locations

In [ ]:
print("Cluster naming run:")
print(RUN_DIR.as_posix())

print("\nCluster name suggestions:")
print(cluster_name_suggestions_path.as_posix())

if CREATE_RENAME_PLAN:
    print("\nFolder rename plan:")
    print(folder_rename_plan_path.as_posix())

if CREATE_NAMED_OUTPUT:
    print("\nNamed output directory:")
    print(NAMED_OUTPUT_DIR.as_posix())
else:
    print("\nNamed output was not materialized. Set CREATE_NAMED_OUTPUT = True if you want to create it.")